# Phase 3 — IrisNet Architecture Verification

This notebook verifies that the IrisNet feature extractor and ArcFace classification head build and behave correctly.

**Checks performed:**
1. `build_irisnet()` constructs without errors and prints a full `model.summary()`.
2. `ArcFaceLayer` instantiates correctly.
3. A single forward pass on a random `(1, 128, 128, 1)` tensor produces an output of shape `(1, 512)`.
4. The L2-norm of the output embedding is exactly `1.0` (proves the Lambda normalisation layer works).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import tensorflow as tf

from src.models.architecture import build_irisnet
from src.models.arcface_loss import ArcFaceLayer

print('TensorFlow version :', tf.__version__)
print('NumPy version      :', np.__version__)

In [ ]:
# Build IrisNet and print full layer summary
model = build_irisnet(input_shape=(128, 128, 1), embedding_dim=512)
model.summary()

In [ ]:
# Instantiate ArcFace head (weights are created lazily on first call)
arcface = ArcFaceLayer(num_classes=1000, embedding_dim=512)
print('ArcFaceLayer instantiated successfully.')
print(f'  margin = {arcface.margin}, scale = {arcface.scale}')

In [ ]:
# Forward pass + assertions
np.random.seed(42)
dummy = np.random.rand(1, 128, 128, 1).astype(np.float32)

embedding = model(dummy, training=False)

# Shape assertion
assert embedding.shape == (1, 512), f'Expected (1, 512), got {embedding.shape}'

# L2-norm assertion — must be 1.0 to prove the Lambda normalisation layer is active
l2_norm = float(tf.norm(embedding, axis=1).numpy()[0])
assert abs(l2_norm - 1.0) < 1e-5, f'L2 norm should be 1.0, got {l2_norm:.8f}'

print(f'Output shape  : {embedding.shape}   ✓')
print(f'L2 norm       : {l2_norm:.8f}        ✓')
print(f'Total params  : {model.count_params():,}')